In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from sklearn.ensemble    import RandomForestClassifier
from sklearn.metrics     import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval
from utils.feature_engineering import add_statistical_features, drop_highly_correlated_features


In [7]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()


In [13]:
def undersample(X, y, ratio, random_state=23):
    """Random undersampling of majority class to achieve ratio (pos:neg = 1:ratio)
       X: DataFrame, y: Series with index aligned to X
    """
    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg_sample = int(n_pos * ratio)
    if n_neg_sample > len(neg_idx):
        raise ValueError(f"Requested neg samples {n_neg_sample} > available {len(neg_idx)}")

    neg_sampled_idx = pd.Series(neg_idx).sample(n=n_neg_sample, random_state=random_state).values

    selected_idx = np.concatenate([pos_idx.values, neg_sampled_idx])
    np.random.RandomState(seed=random_state).shuffle(selected_idx)

    X_bal = X.loc[selected_idx].reset_index(drop=True)
    y_bal = y.loc[selected_idx].reset_index(drop=True)

    return X_bal, y_bal

In [9]:
# ID와 Target 분리하기 
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [ ]:
# zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99, False)


In [15]:
print(X_features.shape)

# underSampling
ratio = 10
X_Balance, y_balance = undersample(X_features, y_labels, ratio)
print(X_Balance.shape, y_balance.shape)

(76020, 149)
(33088, 149) (33088,)


In [16]:
# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [17]:
# 상관계수 높은 feature들 삭제하기
X_reduced, to_drop = drop_highly_correlated_features(X_Balance)
X_test_reduced = X_test.drop(to_drop, axis=1)


In [18]:
print(X_reduced.shape, X_test_reduced.shape)

(33088, 92) (75818, 92)


In [25]:
# 삭제된 컬럼 개수 확인
print("Train에서 삭제된 컬럼 개수:", len(to_drop))

Train에서 삭제된 컬럼 개수: 57


In [ ]:
for i, col in enumerate(sorted(to_drop), start=1):
    print(f"{i:>2} : {col}")


In [20]:
# 리스트를 Pandas Series로 변환
series = pd.Series(sorted(to_drop), name="Dropped_Columns")

# CSV 파일로 저장
series.to_csv("../data/99perCorr95UnderSampling10DroppedColumns_20251122.xls", index=False)


In [ ]:
# 스케일링 
# X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)

In [23]:
# 학습/테스트 데이터 분리
# X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
X_train, X_val, y_train, y_val = data_split(X_reduced, y_balance)

In [ ]:
# model_name = 'RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# model_name = 'RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
model_name = 'RandomForest_99per_corrTh95UnderSampling10_BestOpt' 
# Best Option 적용
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  min_samples_leaf  = 1, 
  min_samples_split = 7,
  n_jobs            = -1 # 병렬처리 여부   
  
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
result_text = '''
✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling10_BestOpt.pkl
  파일 크기: 66.40 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8255, 정확도: 0.9004, 정밀도: 0.4300, 재현율: 0.2907, F1: 0.3469
오차행렬:
[[5784  232]
 [ 427  175]]
실행 시간: 3.1548142433166504
'''

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling10_BestOpt.pkl
  파일 크기: 66.40 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8255, 정확도: 0.9004, 정밀도: 0.4300, 재현율: 0.2907, F1: 0.3469
오차행렬:
[[5784  232]
 [ 427  175]]
실행 시간: 3.1548142433166504


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      57                0.8255    0.3469    0.2907   (UnderSampling 10%)
# 0.95      53                0.842169  0.016287  0.008306 (100% data Not Scaled)
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)


In [ ]:
# # 비교 시각화 
# import pickle
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix


# # 2. 예측 확률 구하기 (y_true는 실제 라벨)
# y_pred_proba_53 = model_53.predict_proba(X_test)[:, 1]
# y_pred_proba_57 = model_57.predict_proba(X_test)[:, 1]

# # 3. ROC, PR, Confusion Matrix 시각화 함수
# def plot_all(y_true, y_pred_proba, threshold=0.95, title="Model"):
#     fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
#     roc_auc = auc(fpr, tpr)

#     precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

#     y_pred = (y_pred_proba >= threshold).astype(int)
#     cm = confusion_matrix(y_true, y_pred)

#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#     # ROC Curve
#     axes[0].plot(fpr, tpr, color="blue", lw=2, label=f"AUC = {roc_auc:.3f}")
#     axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--")
#     axes[0].set_title(f"ROC Curve - {title}")
#     axes[0].set_xlabel("False Positive Rate")
#     axes[0].set_ylabel("True Positive Rate")
#     axes[0].legend(loc="lower right")

#     # Precision-Recall Curve
#     axes[1].plot(recall, precision, color="green", lw=2)
#     axes[1].set_title(f"Precision-Recall Curve - {title}")
#     axes[1].set_xlabel("Recall")
#     axes[1].set_ylabel("Precision")

#     # Confusion Matrix
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2])
#     axes[2].set_title(f"Confusion Matrix - {title}")
#     axes[2].set_xlabel("Predicted")
#     axes[2].set_ylabel("Actual")

#     plt.tight_layout()
#     plt.show()

# # 4. 두 모델 비교 실행
# plot_all(y_true, y_pred_proba_53, threshold=0.95, title="Dropped 53 Features")
# plot_all(y_true, y_pred_proba_57, threshold=0.95, title="Dropped 57 Features")